<a href="https://colab.research.google.com/github/IAT-ComputationalCreativity-Spring2025/Week3-Rule-Based-Systems/blob/main/l-systems_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L-Systems (Lindenmayer Systems) Lab

## Introduction

L-Systems are parallel rewriting systems that were introduced by Aristid Lindenmayer in 1968. They are particularly useful for modeling plant growth and generating fractals. In this lab, we'll explore how to create various patterns using L-Systems.

### Key Concepts:
- **Axiom**: The initial state/string
- **Production Rules**: Rules that define how to replace characters
- **Iterations**: Number of times to apply the rules
- **Turtle Graphics**: System for visualizing the L-System output

Let's start by importing our required libraries:

In [25]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
! pip install ColabTurtle

In [27]:
import ColabTurtle.Turtle as t
from IPython.display import clear_output

## Part 1: Basic L-System Implementation

First, let's implement our core L-System functions. These will be used throughout the lab to generate and draw various patterns.

In [56]:
def create_l_system(iterations, axiom, rules):
    """Generate L-System instructions based on axiom and rules."""
    result = axiom
    for _ in range(iterations):
        new_string = ""
        for char in result:
            new_string += rules.get(char, char)
        result = new_string
    return result


def draw_circle(radius):
  # I want to draw a small circle at the end of the lines
    """Draw a circle using small forward steps and rotations."""
    circumference = 2 * 3.14159 * radius  # Approximate circumference
    step_size = circumference / 10  # Divide circle into 36 steps
    for _ in range(10):
        t.forward(step_size)
        t.right(36)  # Rotate by 10 degrees (360 / 36 = 10)

def draw_l_system(instructions, angle, distance, color_gradient=False, thickness_variation=False):
    """Draw the L-System using turtle graphics.

    Parameters:
    - instructions: string of L-System commands
    - angle: turning angle in degrees
    - distance: forward movement distance
    - color_gradient: whether to apply a color gradient
    - thickness_variation: whether to vary line thickness
    """
    stack = []
    color_step = 0
    thickness = 1 if not thickness_variation else 5  # Starting thickness

    for cmd in instructions:
        if cmd == 'F':  # Move forward and draw
            if color_gradient:
                # Ensure color_step stays within [0, 1]
                color_step = min(1, max(0, color_step))
                # Interpolate between brown and green
                brown = (75, 72, 42)  # Brown RGB
                green = (82, 188, 64)    # Green RGB
                red = int(brown[0] + (green[0] - brown[0]) * color_step)
                green_val = int(brown[1] + (green[1] - brown[1]) * color_step)
                blue = int(brown[2] + (green[2] - brown[2]) * color_step)
                # Clamp RGB values to [0, 255]
                red = min(255, max(0, red))  # Clamp red
                green_val = min(255, max(0, green_val))  # Clamp green
                blue = min(255, max(0, blue))  # Clamp blue
                t.pencolor((red, green_val, blue))  # Set pen color
                color_step += 0.01  # Increment color step
            if thickness_variation:
                thickness = max(3, thickness * 0.95)
                t.pensize(int(thickness))  # Convert thickness to an integer
            t.forward(distance)
        elif cmd == 'f':  # Move forward without drawing
            t.penup()
            t.forward(distance)
            t.pendown()
        elif cmd == '+':  # Turn right
            t.right(angle + random.uniform(-5, 5))  # Add randomness
        elif cmd == '-':  # Turn left
            t.left(angle + random.uniform(-5, 5))  # Add randomness
        elif cmd == '[':  # Save current state
            stack.append((t.position(), t.heading(), t.pensize(), t.pencolor()))
        elif cmd == ']':  # Restore previous state
            position, heading, size, color = stack.pop()
            t.penup()
            # Ensure the new position is non-negative
            x, y = position
            x = max(0, x)  # Clamp x to non-negative
            y = max(0, y)  # Clamp y to non-negative
            t.goto(x, y)
            t.setheading(heading)
            t.pensize(size)
            t.pencolor(color)
            t.pendown()

            if random.random() < 0.8:  # 80% chance to draw a circle
              t.pencolor((255, 192, 203))  # Set pen color to pink + size = 1 so the circle width is the same through out
              t.pensize(1)
              draw_circle(3)  # Draw a small circle
              t.pensize
              t.pencolor(color)  # Restore the original color+pen size



def setup_turtle():
    """Initialize the turtle."""
    t.initializeTurtle()
    t.hideturtle()
    t.speed(13)  # Fastest speed
    t.penup()
    t.goto(200, 400)  # Start at a position far from the edges
    t.pendown()

IndentationError: expected an indented block after 'if' statement on line 79 (<ipython-input-56-c497842b8cdd>, line 80)

In [57]:
feather_axiom = "F"
feather_rules = {"F": "F[+F]F[-F][F]"}  # Stem with barbs branching out
feather_iterations = 3  # Controls the level of recursion
feather_angle = 16.5  # Angle between the stem and barbs
feather_distance = 20  # Length of each segment

setup_turtle()
t.pensize(2)
feather_instructions = create_l_system(feather_iterations, feather_axiom, feather_rules)
draw_l_system(feather_instructions, feather_angle, feather_distance, color_gradient=True, thickness_variation=True)

## Exercise 1: Koch Curve

The Koch curve is a classic example of a fractal pattern. Let's create it using our L-System:

In [ ]:
# Koch curve parameters
koch_axiom = "F"
koch_rules = {"F": "F+F-F-F+F"}
koch_iterations = 3
koch_angle = 90

# Generate and draw
setup_turtle()
t.pensize(2)
koch_instructions = create_l_system(koch_iterations, koch_axiom, koch_rules)
draw_l_system(koch_instructions, koch_angle, 5)

### Exercise 1 Tasks:
1. Try modifying the number of iterations (start with small numbers like 2-4)
2. Change the angle to 60 degrees and observe the difference
3. Modify the rules to create your own variation
4. Try changing the distance parameter

Note: Be careful with high iteration numbers as they can create very complex patterns!

## Exercise 2: Plant Generation

Now let's create a more complex L-System that generates plant-like structures. This system uses brackets to create branches:

In [ ]:
# Plant parameters
plant_axiom = "X"
plant_rules = {
    "X": "F+[[X]-X]-F[-FX]+X",
    "F": "FF"
}
plant_iterations = 4
plant_angle = 25

# Generate and draw
setup_turtle()
t.pensize(1)
plant_instructions = create_l_system(plant_iterations, plant_axiom, plant_rules)
draw_l_system(plant_instructions, plant_angle, 10)

## Challenge: Create a Fractal Tree

Now it's your turn to experiment! Below is a template for creating a fractal tree. Try modifying the parameters to create different tree shapes:

In [ ]:
# Your fractal tree parameters
tree_axiom = "F"  # Start with a single trunk
tree_rules = {"F": "F[+F]F[-F]F"}  # Basic branching rule
tree_iterations = 3
tree_angle = 30

# Generate and draw
setup_turtle()
t.pensize(2)
tree_instructions = create_l_system(tree_iterations, tree_axiom, tree_rules)
draw_l_system(tree_instructions, tree_angle, 10)

### Challenge Tasks:
1. Modify the rules to create more realistic branching
2. Add different colors for different parts of the tree
3. Try to create a tree with varying branch lengths
4. Experiment with asymmetric branching patterns

## Additional Experiments
Here are some ideas for further exploration:
- Create a snowflake pattern
- Generate a spiral pattern
- Implement a dragon curve
- Create a forest of different trees

Tips:
- Higher iterations create more complex patterns but take longer to draw
- Small changes in rules can create dramatically different results
- The angle parameter greatly affects the final appearance
- Remember to close turtle windows between experiments